# **Perfilado de jugadores según su posición (PCA + *clustering*)**

En este estudio se realiza un **perfilado de jugadores según su posición** a partir de los datos agregados de la temporada 2025/26 de todos los jugadores disponibles en la base de datos del TFM.

Se combinan dos técnicas de aprendizaje no supervisado:

1. ***Principal Component Analysis* (PCA)** para reducir la dimensionalidad de las estadísticas y entender en qué destaca cada perfil.
2. ***Clustering* (K-means)** para agrupar a los jugadores en perfiles diferenciados dentro de cada posición.

---

## **1. Tratamiento inicial de los datos**

Antes de modelar se preparan los datos:

- **Filtro de minutos.** Solo se conservan jugadores con **≥ 500 minutos** en la temporada, para que las estadísticas agregadas sean representativas.
- **Agrupación de posiciones.** Los laterales derecho e izquierdo se unen en `FB`, y los extremos en `WG` (no se diferencia por banda). Las posiciones finales son: `GK`, `CB`, `FB`, `DM`, `AM`, `WG`, `ST`.
- **Selección de variables por posición.** Para cada posición se eligen las columnas relevantes (definidas en `utils/cols_to_study_per_position.json`), ya que las métricas que describen a un portero no son las mismas que las de un delantero.
- **Imputación de nulos.** Los valores ausentes se rellenan con la **media** de la variable, para no perder jugadores por datos puntuales faltantes.

El resultado es un *dataframe* de estadísticas por cada una de las 7 posiciones.

In [2]:
import pandas as pd
import os
import numpy as np
import json
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from mplsoccer import Pitch

warnings.filterwarnings("ignore")

DATA_PATH = r"D:\data"
SILVER_DATA_PATH = os.path.join(DATA_PATH, "silver")

# Lectura de dataframes con los que vamos a trabajar
player_info_df = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_2", "player_clean_2.csv"), sep=";")
player_stats_df = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "agg_player.csv"), sep=";")

# Lectura del JSON con las columnas a tener en cuenta por posición
cols_json_path = os.path.join(os.getcwd(), "utils", "cols_to_study_per_position.json")
with open(cols_json_path, "r", encoding="utf-8") as f:
    cols_to_study_per_position = json.load(f)

# Lectura del JSON con los nombres a asignar de PCA
pca_names_json_path = os.path.join(os.getcwd(), "utils", "pca_names.json")
with open(pca_names_json_path, "r", encoding="utf-8") as f:
    pca_names = json.load(f)

In [7]:
# Limpieza inicial - seleccionamos aquellos jugadores que hayan jugado un mínimo de 500 minutos en toda la temporada
cleaned_stats_df = player_stats_df[player_stats_df["MinutesPlayed"]>=500].copy()

# Estructura de posiciones - en los laterales y extremos, los agrupamos (no diferenciamos por lado)
cleaned_stats_df["Position"] = np.where(cleaned_stats_df["Position"].isin(["RB", "LB"]), "FB", cleaned_stats_df["Position"])
cleaned_stats_df["Position"] = np.where(cleaned_stats_df["Position"].isin(["RW", "LW"]), "WG", cleaned_stats_df["Position"])

# Definir "Player" como indice
cleaned_stats_df = cleaned_stats_df.set_index("Player")

# Insertar el valor average a los valores nulos
cleaned_stats_df = cleaned_stats_df.fillna(cleaned_stats_df.mean(numeric_only=True))

# Diccionario para el guardado de los dataframes por posición
dict_position_stats = {}

# Para cada posición, filtraremos el df, seleccionaremos aquellas columnas según cols_to_study_per_position
for position in ["GK","CB","FB","DM","AM","WG","ST"]:
    pos_df = cleaned_stats_df[cleaned_stats_df["Position"] == position].copy()      # Filtrado por posición
    list_to_select = cols_to_study_per_position[position]                           # Selección de aquellas columnas que nos interesan por posición
    pos_df = pos_df[[col for col in list_to_select if col in pos_df.columns]]       # Aplicamos columnas a seleccionar
    dict_position_stats[position] = pos_df                                          # Añadimos al diccionario

---

## **2. Reducción dimensional — *Principal Component Analysis***

Cada posición tiene decenas de estadísticas correlacionadas entre sí. El **PCA** transforma ese conjunto de variables correlacionadas en un conjunto menor de variables **no correlacionadas** —los *componentes principales*— ordenadas de forma que cada una captura la **máxima varianza** posible no explicada por las anteriores.

### **2.0. Fundamento matemático**

**1. Estandarización.** Como las estadísticas tienen escalas muy distintas (porcentajes, acciones por 90 minutos, distancias…), se escala cada variable a media 0 y desviación típica 1 mediante el *z-score*:

$$
z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

Donde $x_{ij}$ es el valor de la variable $j$ para el jugador $i$, $\mu_j$ su media y $\sigma_j$ su desviación típica. Sin este paso, las variables con mayor magnitud dominarían artificialmente los componentes.

**2. Matriz de covarianza.** Sobre la matriz estandarizada $Z$ ($n$ jugadores × $p$ variables) se calcula:

$$
C = \frac{1}{n-1} Z^{\top} Z
$$

Como los datos están estandarizados, $C$ coincide con la **matriz de correlación** entre variables.

**3. Vectores y valores propios.** PCA diagonaliza la matriz de covarianza:

$$
C \, v_k = \lambda_k \, v_k
$$

Cada **vector propio** $v_k$ define la dirección de un componente principal; su **valor propio** $\lambda_k$ mide la varianza que captura. Los componentes se ordenan de mayor a menor $\lambda_k$.

**4. Proyección y *loadings*.** Las puntuaciones de cada jugador en los nuevos ejes se obtienen proyectando los datos sobre los vectores propios: $\text{PC} = Z V$. Los ***loadings*** son las componentes de los vectores propios (`pca.components_.T`): indican **cuánto pesa cada variable original en cada componente** y son la clave para interpretar y nombrar cada eje.

**5. Varianza explicada.** La proporción de información que conserva cada componente es:

$$
\text{Var. explicada}_k = \frac{\lambda_k}{\sum_{j=1}^{p}\lambda_j}
$$

En este estudio se retienen los **5 primeros componentes** ($n\_components = 5$) por posición, que resumen la mayor parte de la varianza con solo 5 ejes interpretables.

### **2.1. Estudio de los *loadings* por posición**

Para cada posición se analizan los pesos (*loadings*) de cada variable en cada componente y se le asigna un **nombre interpretativo**. Por ejemplo, en porteros:

- **PC1 — Build-up:** participación en la salida de balón.
- **PC2 — Directness:** juego directo vs. control con balón.
- **PC3 — Impact:** capacidad de parada y acciones defensivas activas.
- **PC4 — Activity:** volumen de intervenciones / exposición.
- **PC5 — AerialControl:** dominio del juego aéreo y estilo de intervención.

Este mismo análisis se realiza para CB, FB, DM, AM, WG y ST. Los nombres asignados a cada componente y posición se guardan en `utils/pca_names.json` y se aplican como renombrado de columnas para trabajar con ejes interpretables en lugar de `PC1…PC5`.

In [9]:
# Función para el aplicado de PCA
def apply_pca(df: pd.DataFrame, n_components: int) -> pd.DataFrame:

    # Nos quedamos con variables numéricas
    X = df.select_dtypes(include="number").copy()

    # Escalado estándar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # PCA
    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    # Resultado
    pca_df = pd.DataFrame(X_pca, index=df.index, columns=[f"PC{i+1}" for i in range(n_components)])

    # Loadings (peso de cada variable en cada componente)
    loadings_df = pd.DataFrame(pca.components_.T, index=X.columns, columns=[f"PC{i+1}" for i in range(n_components)])

    return pca_df, loadings_df

In [10]:
# Diccionarios para guardar pca df y loadings df
pca_dict = {}
loadings_dict = {}

# Aplicamos pca para cada posición
for position in ["GK","CB","FB","DM","AM","WG","ST"]:
    pos_df = dict_position_stats[position]                              # Buscamos el df por posición
    pca_df, loadings_df = apply_pca(df=pos_df, n_components=5)          # Aplicamos pca
    pca_dict[position] = pca_df                                         # Guardado del dataframe de pca
    loadings_dict[position] = loadings_df                               # Guardado del dataframe de cargas

### **2.1. Estudio de los *loadings* por posición**

A continuación vamos a analizar los pesos que se asignan a cada variable para cada componente principal en cada posición. De esta forma vamos a poder asignar un grupo a cada componente principal por posición, que posteriormente vamos a aplicar para conocer los distintos perfiles de jugadores.

#### **2.1.1. Porteros**

* **PC1 - BuildUp:** Este componente está claramente dominado por la **distribución en corto y participación en salida**, con pesos altos en *PassMeanX*, *PassPercZoneCenter*, *PassPercZoneLeft*, *PassPercZoneRight* y *PassShareDirBackward*, y negativos en *PassMeanLength* y *PassPercDirLeft/Right*. Representa porteros que participan activamente en la construcción desde atrás, con pases más cortos, posicionamiento adelantado en fase de posesión y tendencia a iniciar jugadas organizadas en lugar de juego directo.

* **PC2 - Directness:** Este eje contrapone **juego directo vs control con balón**. Tiene peso positivo en *LongBallShare*, *GoalKicksPer90*, *GoalKicksRate* y *GoalsConcededPer90*, y negativo en *PassAccuracy*, *OwnHalfPassAccuracy* y *LongBallAccuracy*. Define porteros en contextos de juego más directo (mucho envío largo, menor precisión) frente a aquellos en sistemas de posesión con mayor calidad técnica.

* **PC3 - Impact:** Componente que mezcla **capacidad de parada y acciones defensivas activas**. Altos loadings en *SaveRate*, *SavedShotsInsideBoxRate*, *GoalsPreventedDiffPer90*, junto a *KeeperSweeperActionsPer90*, *AccurateKeeperSweeperActionsPer90* y *TouchesPer90*. Describe porteros con alta influencia global en el juego, tanto bajo palos como fuera del área.

* **PC4 - Activity:** Este eje recoge principalmente el **volumen de acciones defensivas y exposición**. Destacan *SavedShotsInsideBoxPer90*, *GoalsConcededPer90*, *PenaltyGoalsConcededPer90*, *TouchesPer90* y *PassAccuracy*. Representa porteros en equipos que conceden más ocasiones, obligándoles a intervenir constantemente.

* **PC5 - AerialControl:** Componente muy marcado por el **dominio del juego aéreo y estilo de intervención**. Altísimo peso en *HighClaimRate* (positivo) y *PunchRate* (negativo), junto a *PunchesPer90*. Diferencia porteros que blocan y dominan centros con seguridad frente a aquellos que despejan más.

In [12]:
display(loadings_dict["GK"])

,PC1,PC2,PC3,PC4,PC5
SaveRate,0.146835,0.188969,0.266035,-1.011730e-01,1.504722e-02
SavesPer90,0.005920,-0.087762,0.133666,3.230558e-01,-5.970326e-02
SavedShotsInsideBoxPer90,-0.028285,-0.047143,0.239276,3.609133e-01,-9.804286e-02
SavedShotsInsideBoxRate,-0.060087,0.032079,0.233020,2.119680e-01,-9.913974e-02
GoalsConcededPer90,-0.127485,-0.223193,-0.208204,2.390716e-01,-5.414796e-02
GoalsPreventedPer90,0.027301,0.085761,0.153669,8.184214e-02,-6.112612e-02
GoalsPreventedDiffPer90,0.118787,0.219738,0.227827,-1.815014e-01,2.790075e-02
CleanSheetsPer90,-0.085060,0.018557,-0.113266,-3.409237e-02,4.089327e-02
PenaltyFacedPer90,0.003266,-0.055087,-0.146901,1.836866e-01,-6.864648e-02
PenaltySavesPer90,-0.012435,0.019445,0.001072,4.674656e-02,-1.103553e-01


#### **2.1.2. Centrales**

* **PC1 - DefensiveVolume:** Este componente está dominado por el **volumen total de acciones defensivas**, con altos valores en *DefensiveActionsPer90*, *InterceptionsPer90*, *BallRecoveriesPer90*, *DuelsWonPer90*, *AerialWonPer90* y *Blocks*. Representa centrales muy activos defensivamente, con gran presencia en todas las fases sin balón.

* **PC2 - Distribution:** Este eje refleja la **capacidad de pase y construcción**, con pesos en *PassAccuracy*, *OwnHalfPassAccuracy*, *PassesPer90*, *PassPercZoneCenter* y *PassPercDirForward*, además de *LongBallAccuracy*. Define centrales con buena salida de balón y participación en la circulación.

* **PC3 - Aggression:** Componente asociado a la **intensidad defensiva y agresividad en la acción**, con loadings en *TacklesPer90*, *TacklesWonPer90*, *FoulsPer90* y *ProgressiveFieldTilt*. Representa centrales que defienden hacia adelante, con mayor tendencia a intervenir activamente.

* **PC4 - Duels:** Este eje recoge la **eficiencia en duelos**, con pesos altos en *DuelWinRate* y *AerialWinRate*, y negativos en errores (*ErrorsLeadToGoal*, *ErrorsLeadToShot*). Define centrales sólidos, fiables y dominantes en enfrentamientos individuales.

* **PC5 - BuildUp:** Componente que diferencia el **estilo de distribución**, con pesos opuestos en *PassShareZoneCenter* vs *PassShareZoneBack*, y negativo en *PassMeanLength*. Representa centrales más posicionales y asociativos frente a perfiles más directos o conservadores.

In [13]:
display(loadings_dict["CB"])

,PC1,PC2,PC3,PC4,PC5
DefensiveActionsPer90,0.223470,-0.247099,0.042753,0.160065,0.028860
DefensiveActionSuccess,-0.130583,-0.161938,0.084038,0.158338,-0.176108
TacklesPer90,0.063299,0.036387,-0.313703,0.139737,-0.156777
TacklesWonPer90,0.047172,0.030967,-0.309745,0.139980,-0.192042
TackleAccuracy,-0.042375,0.003219,-0.022892,-0.006223,-0.099452
InterceptionsPer90,0.244824,-0.016139,-0.154196,0.049786,-0.064250
InterceptionShare,0.190821,0.119117,-0.190633,-0.036673,-0.076924
ClearancesPer90,0.122412,-0.303047,0.188910,0.157439,0.058215
ClearanceShare,-0.123033,-0.220456,0.320060,0.077451,0.066545
ClearanceOffLinePer90,0.069990,0.000653,0.036240,-0.117514,0.312837


#### **2.1.3. Laterales**

* **PC1 - Attacking:** Este componente está claramente dominado por la **producción ofensiva desde banda**, con altos valores en *CrossesPer90*, *AccurateCrossesPer90*, *KeyPassesPer90*, *ExpectedAssistsPer90*, *BigChancesCreatedPer90* y *CrossShare*. Representa laterales con gran impacto en campo rival, generadores de ocasiones y muy activos en centros y último tercio.

* **PC2 - Possession:** Este eje refleja la **participación en la circulación y calidad de pase**, con pesos en *TouchesPer90*, *PassAccuracy*, *OppositionHalfPassAccuracy*, *PassesPer90* y *BallRecoveriesPer90*. Define laterales muy involucrados en la construcción y mantenimiento de la posesión.

* **PC3 - Defensive:** Componente centrado en el **volumen de acciones defensivas**, con loadings en *DefensiveActionsPer90*, *InterceptionsPer90*, *ClearancesPer90*, *TacklesPer90* y *TacklesWonPer90*. Representa laterales con mayor impacto defensivo y capacidad de corrección.

* **PC4 - Progression:** Este eje recoge la **capacidad de progresión y generación de ventajas**, con peso en *WasFouledPer90*, *CornersWonPer90*, *ProgressiveFieldTilt* y acciones de desborde. Define laterales profundos, que avanzan metros y fuerzan situaciones en campo rival.

* **PC5 - Duels:** Componente asociado a la **intensidad física y rendimiento en duelos**, con loadings en *TacklesWonPer90*, *DuelWinRate*, *ContestWinRate* y también volumen de pase (*PassesPer90*). Representa laterales con perfil físico, competitivos en disputas y con presencia constante en el juego.

In [14]:
display(loadings_dict["FB"])

,PC1,PC2,PC3,PC4,PC5
DefensiveActionsPer90,-0.068835,0.037478,0.461406,0.028810,-0.104755
DefensiveActionSuccess,-0.015927,-0.045942,0.043748,-0.229291,0.093913
TacklesPer90,-0.044528,0.029734,0.312685,0.249624,-0.321104
TacklesWonPer90,-0.038972,0.023143,0.301405,0.202824,-0.334566
TackleAccuracy,-0.003423,-0.004499,0.041854,-0.074651,-0.103381
InterceptionsPer90,0.049739,0.196486,0.322456,0.075641,0.227171
InterceptionShare,0.118991,0.232028,0.084377,0.077965,0.363749
BallRecoveriesPer90,0.146389,0.273478,0.170633,0.158051,0.147610
BallRecoveryRate,0.183050,0.217149,-0.154848,0.129334,0.210712
ClearancesPer90,-0.098318,-0.068791,0.328397,-0.167912,-0.047146


#### **2.1.4. Centrocampistas defensivos**

* **PC1 - Security:** Este componente está dominado por la **seguridad en posesión**, con pesos negativos en *PassAccuracy*, *PassesPer90*, *OwnHalfPassAccuracy* y *OppositionHalfPassAccuracy*, y positivos en *PossessionLossRate* y *UnsuccessfulTouchRate*. Diferencia laterales seguros con balón frente a aquellos con mayor pérdida e imprecisión.

* **PC2 - Involvement:** Este eje recoge el **volumen de participación**, con altos valores en *BallRecoveriesPer90*, *OppositionHalfPassShare*, *TouchesPerPass*, *PossessionLostPer90* y *DispossessedPer90*. Representa laterales muy involucrados en el juego, tanto en ataque como en transiciones.

* **PC3 - Defensive:** Componente centrado en la **actividad defensiva**, con peso en *DefensiveActionsPer90*, *InterceptionsPer90*, *TacklesPer90* y *OwnHalfPassShare*, y negativo en *ProgressiveFieldTilt*. Define laterales con rol más conservador y defensivo.

* **PC4 - Directness:** Este eje está asociado a un estilo de **juego más directo**, con loadings altos en *PassMeanLength* y negativos en *TacklesPer90*, *FoulsPer90* y pérdidas por conducción. Representa laterales que progresan mediante envíos largos más que por combinación.

* **PC5 - Aggression:** Componente vinculado a la **intensidad en duelos y contacto físico**, con pesos en *TacklesPer90*, *TacklesWonPer90*, *DuelWinRate*, *WasFouledPer90* y orientación de juego lateral (*PassShareZoneLeft/Right*). Representa laterales más agresivos y activos en disputas individuales.

In [15]:
display(loadings_dict["DM"])

,PC1,PC2,PC3,PC4,PC5
DefensiveActionsPer90,-0.020611,-0.093593,0.354985,-0.102173,0.223918
DefensiveActionSuccess,-0.016755,-0.054908,0.086803,0.073267,-0.008555
TacklesPer90,0.010783,-0.063788,0.208385,-0.188677,0.356201
TacklesWonPer90,0.012099,-0.064155,0.203517,-0.168659,0.366320
TackleAccuracy,0.007369,-0.009656,0.053336,-0.013790,0.130480
InterceptionsPer90,0.018894,0.064388,0.344045,-0.046329,0.068938
InterceptionShare,0.057371,0.193252,0.188997,0.035759,-0.102583
BallRecoveriesPer90,0.104059,0.243938,0.243117,0.003292,0.011368
BallRecoveryRate,0.115197,0.285136,-0.053722,0.090072,-0.175279
DuelWinRate,-0.111828,-0.005881,0.130492,0.005299,0.212895


#### **2.1.5. Centrocampistas Ofensivos**

* **PC1 - Output:** Este componente está dominado por la **producción ofensiva total**, con altos valores en *ExpectedGoalsPer90*, *GoalsPer90*, *ShotsOnTargetPer90*, *KeyPassesPer90*, *ExpectedAssistsPer90* y *BigChancesCreatedPer90*. Representa mediapuntas con alto impacto directo en goles y ocasiones.

* **PC2 - Possession:** Este eje recoge la **capacidad de control y circulación**, con pesos altos en *PassAccuracy*, *PassesPer90*, *OppositionHalfPassAccuracy* y distribución por zonas (*PassPercZoneCenter/Right/Left*), y negativo en *PossessionLossRate*. Define jugadores que organizan y mantienen la posesión con seguridad.

* **PC3 - Distance:** Componente asociado a la **distancia de acciones ofensivas**, con peso en *ShotMeanLength* y *PassMeanLength*, y negativo en métricas de eficiencia de tiro (*GoalConversion*, *ShotsOnTargetRate*). Representa mediapuntas que finalizan desde más lejos o con menor eficiencia.

* **PC4 - Finishing:** Este eje está relacionado con la **eficiencia de cara a portería**, con loadings en *GoalConversion*, *ShotAccuracy*, *ShotsOnTargetPer90* y *GoalsPer90*. Diferencia jugadores más precisos y efectivos en la finalización.

* **PC5 - Positioning:** Componente vinculado al **posicionamiento y ocupación de espacios**, con peso en *PassMeanX*, *PassShareZoneCenter*, *ShotShareZoneLeft/Right* y *CornersWonPer90*. Representa mediapuntas que destacan por su lectura espacial y capacidad para aparecer en zonas peligrosas.

In [16]:
display(loadings_dict["AM"])

,PC1,PC2,PC3,PC4,PC5
TouchesPer90,0.034679,0.275994,0.088282,-0.005928,0.015823
PassAccuracy,-0.131004,0.303751,-0.068018,0.035029,-0.061752
PassesPer90,-0.061392,0.310440,0.041065,0.031065,-0.065905
OppositionHalfPassShare,0.188138,0.055812,0.043971,-0.073737,0.102276
OppositionHalfPassAccuracy,-0.088237,0.307912,-0.083887,0.020974,-0.038189
ProgressiveFieldTilt,0.235910,-0.007888,0.027636,-0.056084,0.092117
KeyPassesPer90,0.203567,0.184569,0.173417,0.003787,-0.010858
KeyPassesPerPass,0.252011,-0.007140,0.145655,-0.009981,0.031154
GoalAssistsPer90,0.140890,0.162254,0.113035,-0.040824,0.028483
AssistConversion,0.034498,0.070254,0.029073,-0.045138,0.052053


#### **2.1.5. Extremos**

* **PC1 - Attacking:** Este componente está dominado por la **producción ofensiva y generación desde banda**, con altos valores en *KeyPassesPer90*, *ExpectedAssistsPer90*, *BigChancesCreatedPer90*, *CrossesPer90*, *TouchesPer90* y *OppositionHalfPassShare*. Representa extremos muy participativos en campo rival, capaces de generar ocasiones y volumen ofensivo.

* **PC2 - Finishing:** Este eje refleja la **capacidad de finalización**, con pesos en *GoalsPer90*, *ShotsOnTargetPer90*, *ExpectedGoalsPer90*, *GoalConversion* y *ShotAccuracy*. Define extremos con perfil más goleador y orientado al remate.

* **PC3 - Risk:** Componente asociado al **riesgo en posesión y volumen de duelos ofensivos**, con loadings en *DispossessedPer90*, *DispossessedRate*, *PossessionLossRate* y *ContestsPer90*. Representa extremos que asumen muchos 1vs1, con mayor pérdida pero también mayor desequilibrio.

* **PC4 - Directness:** Este eje recoge la **verticalidad y juego directo**, con pesos negativos en *PassAccuracy* y *PassPercZoneCenter*, y positivos en *PossessionLossRate*, *OffsidesPer90* y longitud de pase (*PassMeanLength*). Define extremos que buscan profundidad constantemente, con menor énfasis en la asociación.

* **PC5 - ShootingProfile:** Componente que diferencia el **tipo de finalización**, con pesos en *ShotMeanLength*, *WasFouledPer90*, *MissedShotRate* y métricas de contacto ofensivo. Representa extremos que finalizan desde distintas zonas y generan faltas en situaciones ofensivas.

In [17]:
display(loadings_dict["WG"])

,PC1,PC2,PC3,PC4,PC5
TouchesPer90,0.182071,0.050338,-0.062161,0.289384,-0.074258
PassAccuracy,0.025351,-0.076083,-0.164695,0.351263,-0.083672
OppositionHalfPassShare,0.222784,-0.031186,0.165621,0.101865,-0.033425
ProgressiveFieldTilt,0.204985,-0.088326,0.174921,-0.023180,-0.066587
GoalsPer90,0.163015,-0.259311,-0.190292,-0.193405,-0.104200
ExpectedGoalsPer90,0.203919,-0.205541,-0.012422,0.005442,0.036990
ExpectedGoalsOnTargetPer90,0.213321,-0.226496,-0.075191,-0.053395,0.035263
GoalConversion,0.082978,-0.213602,-0.239086,-0.219561,-0.050342
GoalsPerShotOnTarget,0.036387,-0.149156,-0.180295,-0.166805,-0.129245
GoalsMinusxG,0.028555,-0.112806,-0.192318,-0.218582,-0.168383


#### **2.1.7. Delanteros**

* **PC1 - Distance:** Este componente está dominado por la **distancia de juego y finalización**, con altos valores en *ShotMeanLength*, *PassMeanLength* y negativos en *ExpectedGoalsPer90* y *ExpectedGoalsOnTargetPer90*. Representa delanteros que finalizan desde posiciones más lejanas o participan en acciones menos cercanas al área.

* **PC2 - Finishing:** Este eje recoge la **eficiencia goleadora**, con pesos muy altos en *GoalsPer90*, *GoalConversion*, *ShotsOnTargetPer90*, *ShotAccuracy* y métricas como *GoalsMinusxG*. Define delanteros con gran capacidad para convertir ocasiones en gol.

* **PC3 - Involvement:** Componente asociado al **volumen de participación ofensiva**, con loadings en *TotalShotsPer90*, *ExpectedGoalsPer90*, *KeyPassesPer90*, *ExpectedAssistsPer90* y *DuelsWonPer90*. Representa delanteros que intervienen mucho en el juego y generan volumen ofensivo.

* **PC4 - Efficiency:** Este eje refleja la **calidad vs desperdicio en el tiro**, con pesos negativos en *MissedShotRate* y positivos en *SavedShotRate*, *ShotAccuracy* y contribución ofensiva. Diferencia delanteros más eficientes frente a otros que desperdician más ocasiones.

* **PC5 - Physicality:** Componente vinculado a la **intensidad física y tipo de finalización**, con peso en *TotalShotsPer90*, *BlockedShotsPer90*, *ShotsOffTargetPer90* y acciones de contacto. Representa delanteros más físicos, que generan volumen mediante presencia y disputa.

In [18]:
display(loadings_dict["ST"])

,PC1,PC2,PC3,PC4,PC5
TouchesPer90,0.003896,-0.068459,0.263660,0.157050,0.155107
GoalsPer90,0.353395,0.108952,0.013181,-0.102287,0.060557
ExpectedGoalsPer90,0.277107,-0.076491,0.202843,0.032371,-0.174434
ExpectedGoalsOnTargetPer90,0.310262,-0.025305,0.153202,0.035451,-0.138992
GoalConversion,0.271924,0.131341,-0.146646,-0.121601,0.072369
GoalsPerShotOnTarget,0.166840,0.068883,-0.074778,-0.170234,0.103959
GoalsMinusxG,0.124001,0.186148,-0.156656,-0.139765,0.254904
GoalsMinusxGOT,0.080251,0.152482,-0.144796,-0.158185,0.258780
TotalShotsPer90,0.229202,-0.018107,0.257874,0.000724,0.014543
ShotsOnTargetPer90,0.345170,0.084219,0.089000,0.013586,-0.014345


### **2.2. Aplicado de los nuevos valores de los componentes principales**

Ahora que tenemos los grupos aplicados, guardamos sus nuevos nombres a un fichero JSON para poderlos aplicar en formato diccionario posteriormente.

In [19]:
# Aplicamos para cada posición
for position in ["GK","CB","FB","DM","AM","WG","ST"]:

    # Dataframes
    pca_df = pca_dict[position]
    loadings_df = loadings_dict[position]

    # Rename de las columnas
    columns_dict = pca_names[position]
    pca_df = pca_df.rename(columns=columns_dict)
    loadings_df = loadings_df.rename(columns=columns_dict)

    # Aplicado al diccionario
    pca_dict[position] = pca_df
    loadings_dict[position] = loadings_df

---

## **3. División en grupos — *Clustering* (K-means)**

A partir de las puntuaciones de PCA, se aplica **K-means** para agrupar a los jugadores de cada posición en perfiles. K-means busca particionar los jugadores en $K$ grupos minimizando la **inercia** (suma de distancias al cuadrado de cada punto a su centroide):

$$
\text{Inertia} = \sum_{k=1}^{K} \sum_{x_i \in C_k} \lVert x_i - \mu_k \rVert^2
$$

Donde $C_k$ es el cluster $k$ y $\mu_k$ su centroide.

### **3.1. Selección del número óptimo de grupos**

Se prueba $K$ de **3 a 6** y se elige el valor que **maximiza el *Silhouette Score***, una métrica que mide cómo de bien separado está cada punto de los demás grupos. Para un punto $i$:

$$
s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}}
$$

Donde:

- $a(i)$ = distancia media de $i$ a los puntos de **su propio** cluster (cohesión),
- $b(i)$ = distancia media de $i$ al cluster vecino **más cercano** (separación).

El *Silhouette Score* global es la media de $s(i)$ y vive en $[-1, 1]$: cuanto más cerca de 1, mejor definida está la estructura de grupos. Se entrena el modelo final con el $K$ ganador y se asigna a cada jugador su cluster.

In [20]:
# Aplica K-means probando diferentes valores de K y selecciona el K óptimo
def apply_kmeans_optimal(df: pd.DataFrame, k_min: int = 3, k_max: int = 6, random_state: int = 42, n_init: int = 20):

    # Nos quedamos solo con variables numéricas
    X = df.select_dtypes(include="number").copy()

    # Eliminamos filas con NaN
    X = X.dropna()

    # Ajustamos k_max para no superar el número de muestras
    k_max = min(k_max, len(X) - 1)

    if k_max < k_min:
        raise ValueError("No hay suficientes filas para aplicar clustering.")

    results = []

    # Probamos diferentes valores de K
    for k in range(k_min, k_max + 1):
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=n_init)
        labels = kmeans.fit_predict(X)
        silhouette = silhouette_score(X, labels)
        results.append({"K": k, "Inertia": kmeans.inertia_, "Silhouette": silhouette})

    scores_df = pd.DataFrame(results)

    # Seleccionamos el mejor K según Silhouette Score
    best_k = scores_df.loc[scores_df["Silhouette"].idxmax(), "K"]

    # Entrenamos el modelo final
    best_model = KMeans(n_clusters=int(best_k), random_state=random_state, n_init=n_init)
    final_labels = best_model.fit_predict(X)

    # Creamos copia del df original
    df_clustered = df.copy()
    df_clustered["Cluster"] = np.nan

    # Asignamos clusters solo a las filas válidas
    df_clustered.loc[X.index, "Cluster"] = final_labels
    df_clustered["Cluster"] = df_clustered["Cluster"].astype("Int64")

    return df_clustered, scores_df

In [21]:
# Diccionarios para guardar clustered df y scores
clustered_dict = {}
scores_k_means_dict = {}

# Aplicamos pca para cada posición
for position in ["GK","CB","FB","DM","AM","WG","ST"]:
    pos_df = pca_dict[position]                                         # Buscamos el df por posición
    df_clustered, scores_df = apply_kmeans_optimal(df=pos_df)           # Aplicamos clustering
    clustered_dict[position] = df_clustered                             # Guardado
    scores_k_means_dict[position] = scores_df                           # Guardado

  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\ASUS\anaconda3\envs\soccer_env\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ASUS\anaconda3\envs\soccer_env\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\ASUS\anaconda3\envs\soccer_env\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^

---

## **4. Análisis de los grupos**

Con los clusters ya formados, se interpreta **qué perfil representa cada grupo** mediante visualizaciones:

- ***Scatter* PC vs. PC** coloreado por cluster, junto a los *loadings* de las variables más relevantes (para ver qué distingue a cada grupo).
- **Mapa de calor de la posición media de pase** sobre el campo, por cluster, usando `PassMeanX` / `PassMeanY`.

A cada cluster se le asigna un **nombre de rol** descriptivo. Algunos ejemplos:

- **Porteros:** `DISTRIBUTOR`, `LINE KEEPER`, `DIRECT OUTLET`.
- **Centrales:** `CIRCULATOR`, `DEEP ANCHOR`, `FRAGILE DEFENDER`, `DUEL SPECIALIST`, `COMPLETE BUILDER`, `PROGRESSIVE PLAYMAKER`.
- **Laterales:** `BALANCED`, `RUNNER`, `DEFENSIVE`.

(Y de forma equivalente para DM, AM, WG y ST.)

Los roles definitivos por posición se guardan para poder aplicarlos a cualquier jugador.

In [22]:
# Creación de la visualización para analizar los componentes principales
def pca_plot(df: pd.DataFrame, loadings_df: pd.DataFrame, x_col: str, y_col: str, title: str = "", top_n_loadings: int = 15):

    # Cluster categórico
    plot_df = df.copy()
    plot_df["Cluster"] = plot_df["Cluster"].astype(str)

    # Seleccionar loadings más relevantes
    loadings_plot = loadings_df.copy()
    loadings_plot["Importance"] = (loadings_plot[x_col].abs() + loadings_plot[y_col].abs())
    loadings_plot = (loadings_plot.sort_values("Importance", ascending=False).head(top_n_loadings).drop(columns="Importance"))

    # Crear subplots
    fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.08)

    # Scatter izquierda
    scatter_fig = px.scatter(plot_df, x=x_col, y=y_col, color="Cluster", hover_name="PlayerName", hover_data={x_col:False,y_col:False,"Cluster":False})
    for trace in scatter_fig.data:
        trace.marker.size = 8
        trace.marker.opacity = 0.8
        fig.add_trace(trace, row=1, col=1)

    # Loadings derecha
    for variable, row_vals in loadings_plot.iterrows():
        fig.add_trace(go.Scatter(x=[0, row_vals[x_col]], y=[0, row_vals[y_col]], mode="lines+markers+text", text=["", variable], textposition="top center",
                                 line=dict(width=2, color="black"), marker=dict(size=5, color="black"), showlegend=False, hoverinfo="skip"), row=1, col=2)

    # Ejes centro loadings
    fig.add_hline(y=0, line_dash="dot", line_color="gray", row=1, col=2)
    fig.add_vline(x=0, line_dash="dot", line_color="gray", row=1, col=2)

    # Ejes centro loadings
    fig.add_hline(y=0, line_dash="dot", line_color="gray", row=1, col=2)
    fig.add_vline(x=0, line_dash="dot", line_color="gray", row=1, col=2)

    # Nombres de ejes
    fig.update_xaxes(title_text=x_col, row=1, col=1)
    fig.update_yaxes(title_text=y_col, row=1, col=1)

    fig.update_xaxes(title_text=x_col, row=1, col=2)
    fig.update_yaxes(title_text=y_col, row=1, col=2)

    # Layout
    fig.update_layout(template="plotly_white", width=1000, height=400, title=dict(text=title, x=0.5, xanchor="center"), legend_title="Cluster")
    
    return fig

In [23]:
# Creación de un campo de fútbol con las medias de pase por jugador según posición
def plot_pitch_clusters(df_pos: pd.DataFrame, title: str = ""):

    # Seleccionamos las estadísticas de pases por jugador (drop na para evitar valores comunes)
    pass_stats = player_stats_df[["Player", "PassMeanX", "PassMeanY"]].copy()
    pass_stats = pass_stats.dropna()

    # Seleccionamos el cluster por jugador
    df_pos = df_pos.reset_index()
    dict_cluster = dict(zip(df_pos["Player"], df_pos["Cluster"]))

    # Aplicamos el cluster al dataframe
    pass_stats["Cluster"] = pass_stats["Player"].map(dict_cluster)
    pass_stats = pass_stats.dropna()

    # Creamos un campo
    pitch = Pitch(pitch_type='custom', pitch_length=100, pitch_width=100, line_color='white', pitch_color='green')
    fig, ax = pitch.draw(figsize=(10, 7))

    # Colores por cluster (ajusta si tienes más)
    colors = {0: 'red', 1: 'blue', 2: 'yellow', 3: 'orange', 4: 'pink'}

    # Plot por cluster
    for cluster in sorted(pass_stats["Cluster"].dropna().unique()):
        subset = pass_stats[pass_stats["Cluster"] == cluster]
        pitch.scatter(subset["PassMeanX"], subset["PassMeanY"], ax=ax, s=50, color=colors.get(cluster, 'white'), label=f'Cluster {int(cluster)}', edgecolors='black')

    ax.legend(loc='upper right')
    ax.set_title(title, fontsize=14)

    return fig, ax

### **4.1. Análisis visual por posición**

Usando la función creada, por posición vamos a analizar los grupos y a definir un nombre y perfil para cada uno.

#### **4.1.1. Porteros**

0. **DISTRIBUTOR**: portero con alta implicación en fase de construcción, destacando en volumen de pases en campo rival y participación activa en la circulación. Combina juego relativamente corto con cierta progresión, manteniendo un impacto defensivo equilibrado sin sobresalir en métricas puras de parada.

1. **LINE KEEPER**: perfil de portero más posicional y reactivo, con baja participación en el build-up y menor volumen de acciones con balón. Su rol se centra en la defensa del área y la portería, con actividad moderada en paradas pero escasa influencia en la generación de juego.

2. **DIRECT OUTLET**: portero orientado a un juego más vertical, con alta implicación en la salida pero priorizando envíos largos y rápidos. Reduce la elaboración en corto, buscando progresar directamente, con menor peso relativo en métricas de impacto defensivo y mayor asociación a contextos de juego directo.

In [24]:
# Diccionario con los nombres de los jugadores
player_names_dict = dict(zip(player_info_df["ID"], player_info_df["Name"]))

# Posición
pos = "GK"

# Seleccionamos posicion y aplicamos nombre del jugador
pos_df = clustered_dict[pos]
pos_df["PlayerName"] = pos_df.index.map(player_names_dict)
loadings_df = loadings_dict[pos]

# Carpeta destino
output_dir = os.path.join(os.getcwd(), "results", "visualizations", pos.lower())

# Si no existe, creamos
if not os.path.exists(output_dir):

    os.makedirs(output_dir)

    # Grupos
    components = loadings_df.columns
    c1 = components[0]
    c2 = components[1]
    c3 = components[2]
    c4 = components[3]
    c5 = components[4]

    # Creación de las visualizaciones
    fig_c1_c2 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c2, title=f"{c1} vs. {c2}")  # PC1-PC2
    fig_c1_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c3, title=f"{c1} vs. {c3}")  # PC1-PC3
    fig_c1_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c4, title=f"{c1} vs. {c4}")  # PC1-PC4
    fig_c1_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c5, title=f"{c1} vs. {c5}")  # PC1-PC5
    fig_c2_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c3, title=f"{c2} vs. {c3}")  # PC2-PC3
    fig_c2_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c4, title=f"{c2} vs. {c4}")  # PC2-PC4
    fig_c2_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c5, title=f"{c2} vs. {c5}")  # PC2-PC5
    fig_c3_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c4, title=f"{c3} vs. {c4}")  # PC3-PC4
    fig_c3_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c5, title=f"{c3} vs. {c5}")  # PC3-PC5
    fig_c4_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c4, y_col=c5, title=f"{c4} vs. {c5}")  # PC4-PC5

    # Nombres para cada gráfico
    names = [f"01_{c1}_{c2}", f"02_{c1}_{c3}", f"03_{c1}_{c4}", f"04_{c1}_{c5}", f"05_{c2}_{c3}", f"06_{c2}_{c4}", 
            f"07_{c2}_{c5}", f"08_{c3}_{c4}", f"09_{c3}_{c5}", f"10_{c4}_{c5}"]

    # Lista
    figs = [fig_c1_c2, fig_c1_c3, fig_c1_c4, fig_c1_c5, fig_c2_c3, fig_c2_c4, fig_c2_c5, fig_c3_c4, fig_c3_c5, fig_c4_c5]
    for fig, name in zip(figs, names):
        file_path = os.path.join(output_dir, f"{name}.png")
        fig.write_image(file_path, width=1200, height=800)

    # Creación del dataframe de pases - solo tenemos pases para el grupo 1
    # fig, ax = plot_pitch_clusters(df_pos=pos_df, title=f"Average pass position of {pos} (clustered)")
    # fig.savefig(os.path.join(output_dir, "PassField.png"), dpi=300, bbox_inches="tight")

#### **4.1.2. Centrales**

0. **CIRCULATOR**: central con rol funcional en salida de balón, participando en la circulación sin asumir grandes riesgos ni destacar en volumen defensivo. Presenta valores intermedios tanto en distribución como en actividad defensiva, actuando como pieza de equilibrio dentro del sistema.

1. **DEEP ANCHOR**: defensor de perfil bajo en build-up y con limitada influencia en fases ofensivas. Su juego se caracteriza por una posición más retrasada y conservadora, con baja agresividad y escasa participación en duelos dominantes, priorizando la protección estructural.

2. **FRAGILE DEFENDER**: central con bajo volumen de acciones defensivas y menor impacto en duelos, asociado a peores métricas de rendimiento defensivo (errores y concesiones). Perfil que sufre en contextos exigentes y con limitada aportación en construcción.

3. **DUEL SPECIALIST**: central altamente agresivo, con fuerte presencia en duelos defensivos y acciones de interrupción. Destaca en tackles, duelos aéreos y actividad defensiva, con menor peso en la construcción de juego y mayor orientación a la defensa directa.

4. **COMPLETE BUILDER**: central moderno con alto volumen defensivo y gran capacidad de participación en salida de balón. Combina recuperación, anticipación e intervención defensiva con una sólida contribución en la progresión del juego, siendo clave en ambos lados del balón.

5. **PROGRESSIVE PLAYMAKER**: central claramente orientado a la construcción, con altos valores en métricas de pase, precisión y progresión. Actúa como iniciador del juego desde zonas retrasadas, con menor énfasis en agresividad defensiva pero gran impacto en la fase ofensiva.

In [25]:
# Diccionario con los nombres de los jugadores
player_names_dict = dict(zip(player_info_df["ID"], player_info_df["Name"]))

# Posición
pos = "CB"

# Seleccionamos posicion y aplicamos nombre del jugador
pos_df = clustered_dict[pos]
pos_df["PlayerName"] = pos_df.index.map(player_names_dict)
loadings_df = loadings_dict[pos]

# Carpeta destino
output_dir = os.path.join(os.getcwd(), "results", "visualizations", pos.lower())

# Si no existe, creamos
if not os.path.exists(output_dir):

    os.makedirs(output_dir)

    # Grupos
    components = loadings_df.columns
    c1 = components[0]
    c2 = components[1]
    c3 = components[2]
    c4 = components[3]
    c5 = components[4]

    # Creación de las visualizaciones
    fig_c1_c2 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c2, title=f"{c1} vs. {c2}")  # PC1-PC2
    fig_c1_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c3, title=f"{c1} vs. {c3}")  # PC1-PC3
    fig_c1_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c4, title=f"{c1} vs. {c4}")  # PC1-PC4
    fig_c1_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c5, title=f"{c1} vs. {c5}")  # PC1-PC5
    fig_c2_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c3, title=f"{c2} vs. {c3}")  # PC2-PC3
    fig_c2_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c4, title=f"{c2} vs. {c4}")  # PC2-PC4
    fig_c2_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c5, title=f"{c2} vs. {c5}")  # PC2-PC5
    fig_c3_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c4, title=f"{c3} vs. {c4}")  # PC3-PC4
    fig_c3_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c5, title=f"{c3} vs. {c5}")  # PC3-PC5
    fig_c4_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c4, y_col=c5, title=f"{c4} vs. {c5}")  # PC4-PC5

    # Nombres para cada gráfico
    names = [f"01_{c1}_{c2}", f"02_{c1}_{c3}", f"03_{c1}_{c4}", f"04_{c1}_{c5}", f"05_{c2}_{c3}", f"06_{c2}_{c4}", 
            f"07_{c2}_{c5}", f"08_{c3}_{c4}", f"09_{c3}_{c5}", f"10_{c4}_{c5}"]

    # Lista
    figs = [fig_c1_c2, fig_c1_c3, fig_c1_c4, fig_c1_c5, fig_c2_c3, fig_c2_c4, fig_c2_c5, fig_c3_c4, fig_c3_c5, fig_c4_c5]
    for fig, name in zip(figs, names):
        file_path = os.path.join(output_dir, f"{name}.png")
        fig.write_image(file_path, width=1200, height=800)

    # Creación del dataframe de pases
    fig, ax = plot_pitch_clusters(df_pos=pos_df, title=f"Average pass position of {pos} (clustered)")
    fig.savefig(os.path.join(output_dir, "PassField.png"), dpi=300, bbox_inches="tight")

#### **4.1.3. Laterales**

0. **BALANCED**: lateral de perfil equilibrado que combina participación en la construcción con una contribución defensiva sólida. Presenta valores intermedios en progresión y attacking, manteniendo un rol estable tanto en salida de balón como en defensa posicional.

1. **RUNNER**: lateral claramente orientado a la fase ofensiva, con alta capacidad de progresión y generación de peligro (centros, key passes, acciones en campo rival). Opera en alturas más avanzadas, aportando amplitud y profundidad, con menor peso relativo en tareas defensivas.

2. **DEFENSIVE**: lateral conservador, con baja implicación en fase ofensiva y menor volumen de progresión. Su juego se desarrolla en zonas más retrasadas, priorizando la solidez defensiva, la protección del bloque y la circulación segura.

In [26]:
# Diccionario con los nombres de los jugadores
player_names_dict = dict(zip(player_info_df["ID"], player_info_df["Name"]))

# Posición
pos = "FB"

# Seleccionamos posicion y aplicamos nombre del jugador
pos_df = clustered_dict[pos]
pos_df["PlayerName"] = pos_df.index.map(player_names_dict)
loadings_df = loadings_dict[pos]

# Carpeta destino
output_dir = os.path.join(os.getcwd(), "results", "visualizations", pos.lower())

# Si no existe, creamos
if not os.path.exists(output_dir):

    os.makedirs(output_dir)

    # Grupos
    components = loadings_df.columns
    c1 = components[0]
    c2 = components[1]
    c3 = components[2]
    c4 = components[3]
    c5 = components[4]

    # Creación de las visualizaciones
    fig_c1_c2 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c2, title=f"{c1} vs. {c2}")  # PC1-PC2
    fig_c1_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c3, title=f"{c1} vs. {c3}")  # PC1-PC3
    fig_c1_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c4, title=f"{c1} vs. {c4}")  # PC1-PC4
    fig_c1_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c5, title=f"{c1} vs. {c5}")  # PC1-PC5
    fig_c2_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c3, title=f"{c2} vs. {c3}")  # PC2-PC3
    fig_c2_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c4, title=f"{c2} vs. {c4}")  # PC2-PC4
    fig_c2_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c5, title=f"{c2} vs. {c5}")  # PC2-PC5
    fig_c3_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c4, title=f"{c3} vs. {c4}")  # PC3-PC4
    fig_c3_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c5, title=f"{c3} vs. {c5}")  # PC3-PC5
    fig_c4_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c4, y_col=c5, title=f"{c4} vs. {c5}")  # PC4-PC5

    # Nombres para cada gráfico
    names = [f"01_{c1}_{c2}", f"02_{c1}_{c3}", f"03_{c1}_{c4}", f"04_{c1}_{c5}", f"05_{c2}_{c3}", f"06_{c2}_{c4}", 
            f"07_{c2}_{c5}", f"08_{c3}_{c4}", f"09_{c3}_{c5}", f"10_{c4}_{c5}"]

    # Lista
    figs = [fig_c1_c2, fig_c1_c3, fig_c1_c4, fig_c1_c5, fig_c2_c3, fig_c2_c4, fig_c2_c5, fig_c3_c4, fig_c3_c5, fig_c4_c5]
    for fig, name in zip(figs, names):
        file_path = os.path.join(output_dir, f"{name}.png")
        fig.write_image(file_path, width=1200, height=800)

    # Creación del dataframe de pases
    fig, ax = plot_pitch_clusters(df_pos=pos_df, title=f"Average pass position of {pos} (clustered)")
    fig.savefig(os.path.join(output_dir, "PassField.png"), dpi=300, bbox_inches="tight")

#### **4.1.4. Centrocampistas defensivos**

0. **BALL WINNER**: centrocampista de perfil defensivo e intenso, orientado a la recuperación y la interrupción del juego rival. Destaca en métricas de agresividad y acciones defensivas (tackles, duelos, intercepciones), con menor peso en la construcción y menor seguridad en la circulación.

1. **DEEP PLAYMAKER**: mediocentro con alta participación en el juego y capacidad para organizar la posesión. Combina volumen de intervenciones con progresión y distribución, siendo clave en la salida de balón y en la conexión entre líneas, manteniendo un equilibrio defensivo adecuado.

2. **ANCHOR**: pivote posicional y conservador, enfocado en mantener la estructura del equipo y asegurar la circulación. Presenta alta seguridad en el pase, bajo riesgo y menor volumen de participación ofensiva, actuando como punto de equilibrio y protección del bloque defensivo.

In [27]:
# Diccionario con los nombres de los jugadores
player_names_dict = dict(zip(player_info_df["ID"], player_info_df["Name"]))

# Posición
pos = "DM"

# Seleccionamos posicion y aplicamos nombre del jugador
pos_df = clustered_dict[pos]
pos_df["PlayerName"] = pos_df.index.map(player_names_dict)
loadings_df = loadings_dict[pos]

# Carpeta destino
output_dir = os.path.join(os.getcwd(), "results", "visualizations", pos.lower())

# Si no existe, creamos
if not os.path.exists(output_dir):

    os.makedirs(output_dir)

    # Grupos
    components = loadings_df.columns
    c1 = components[0]
    c2 = components[1]
    c3 = components[2]
    c4 = components[3]
    c5 = components[4]

    # Creación de las visualizaciones
    fig_c1_c2 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c2, title=f"{c1} vs. {c2}")  # PC1-PC2
    fig_c1_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c3, title=f"{c1} vs. {c3}")  # PC1-PC3
    fig_c1_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c4, title=f"{c1} vs. {c4}")  # PC1-PC4
    fig_c1_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c5, title=f"{c1} vs. {c5}")  # PC1-PC5
    fig_c2_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c3, title=f"{c2} vs. {c3}")  # PC2-PC3
    fig_c2_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c4, title=f"{c2} vs. {c4}")  # PC2-PC4
    fig_c2_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c5, title=f"{c2} vs. {c5}")  # PC2-PC5
    fig_c3_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c4, title=f"{c3} vs. {c4}")  # PC3-PC4
    fig_c3_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c5, title=f"{c3} vs. {c5}")  # PC3-PC5
    fig_c4_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c4, y_col=c5, title=f"{c4} vs. {c5}")  # PC4-PC5

    # Nombres para cada gráfico
    names = [f"01_{c1}_{c2}", f"02_{c1}_{c3}", f"03_{c1}_{c4}", f"04_{c1}_{c5}", f"05_{c2}_{c3}", f"06_{c2}_{c4}", 
            f"07_{c2}_{c5}", f"08_{c3}_{c4}", f"09_{c3}_{c5}", f"10_{c4}_{c5}"]

    # Lista
    figs = [fig_c1_c2, fig_c1_c3, fig_c1_c4, fig_c1_c5, fig_c2_c3, fig_c2_c4, fig_c2_c5, fig_c3_c4, fig_c3_c5, fig_c4_c5]
    for fig, name in zip(figs, names):
        file_path = os.path.join(output_dir, f"{name}.png")
        fig.write_image(file_path, width=1200, height=800)

    # Creación del dataframe de pases
    fig, ax = plot_pitch_clusters(df_pos=pos_df, title=f"Average pass position of {pos} (clustered)")
    fig.savefig(os.path.join(output_dir, "PassField.png"), dpi=300, bbox_inches="tight")

#### **4.1.5. Centrocampistas ofensivos**

0. **HYBRID**: jugador de menor impacto ofensivo directo, con participación más equilibrada pero sin destacar especialmente en creación ni en finalización. Actúa como pieza de conexión y apoyo en la fase ofensiva, contribuyendo al juego sin ser el foco principal de generación de ocasiones.

1. **CREATOR**: centrocampista ofensivo orientado a la generación de juego y creación de oportunidades. Destaca en métricas como key passes, expected assists y big chances created, siendo el principal generador de ventajas ofensivas para el equipo.

2. **SCORER**: mediapunta con fuerte orientación al gol, caracterizado por un alto volumen de tiros, goles y expected goals. Tiene menor peso en la creación pura y mayor protagonismo en la finalización de las jugadas.

In [28]:
# Diccionario con los nombres de los jugadores
player_names_dict = dict(zip(player_info_df["ID"], player_info_df["Name"]))

# Posición
pos = "AM"

# Seleccionamos posicion y aplicamos nombre del jugador
pos_df = clustered_dict[pos]
pos_df["PlayerName"] = pos_df.index.map(player_names_dict)
loadings_df = loadings_dict[pos]

# Carpeta destino
output_dir = os.path.join(os.getcwd(), "results", "visualizations", pos.lower())

# Si no existe, creamos
if not os.path.exists(output_dir):

    os.makedirs(output_dir)

    # Grupos
    components = loadings_df.columns
    c1 = components[0]
    c2 = components[1]
    c3 = components[2]
    c4 = components[3]
    c5 = components[4]

    # Creación de las visualizaciones
    fig_c1_c2 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c2, title=f"{c1} vs. {c2}")  # PC1-PC2
    fig_c1_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c3, title=f"{c1} vs. {c3}")  # PC1-PC3
    fig_c1_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c4, title=f"{c1} vs. {c4}")  # PC1-PC4
    fig_c1_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c5, title=f"{c1} vs. {c5}")  # PC1-PC5
    fig_c2_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c3, title=f"{c2} vs. {c3}")  # PC2-PC3
    fig_c2_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c4, title=f"{c2} vs. {c4}")  # PC2-PC4
    fig_c2_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c5, title=f"{c2} vs. {c5}")  # PC2-PC5
    fig_c3_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c4, title=f"{c3} vs. {c4}")  # PC3-PC4
    fig_c3_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c5, title=f"{c3} vs. {c5}")  # PC3-PC5
    fig_c4_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c4, y_col=c5, title=f"{c4} vs. {c5}")  # PC4-PC5

    # Nombres para cada gráfico
    names = [f"01_{c1}_{c2}", f"02_{c1}_{c3}", f"03_{c1}_{c4}", f"04_{c1}_{c5}", f"05_{c2}_{c3}", f"06_{c2}_{c4}", 
            f"07_{c2}_{c5}", f"08_{c3}_{c4}", f"09_{c3}_{c5}", f"10_{c4}_{c5}"]

    # Lista
    figs = [fig_c1_c2, fig_c1_c3, fig_c1_c4, fig_c1_c5, fig_c2_c3, fig_c2_c4, fig_c2_c5, fig_c3_c4, fig_c3_c5, fig_c4_c5]
    for fig, name in zip(figs, names):
        file_path = os.path.join(output_dir, f"{name}.png")
        fig.write_image(file_path, width=1200, height=800)

    # Creación del dataframe de pases
    fig, ax = plot_pitch_clusters(df_pos=pos_df, title=f"Average pass position of {pos} (clustered)")
    fig.savefig(os.path.join(output_dir, "PassField.png"), dpi=300, bbox_inches="tight")

#### **4.1.6. Extremos**

0. **EXPLOSIVE**: extremo con alto volumen ofensivo y juego vertical. Destaca por su capacidad de desborde, centros y acciones directas hacia portería, asumiendo un elevado riesgo que se refleja en pérdidas de balón. Es un perfil orientado a generar ventajas a través del uno contra uno.

1. **FINISHER**: extremo con clara orientación a la finalización. Presenta altos valores en goles, expected goals y volumen de tiros, participando frecuentemente en zonas de remate. Tiene menor protagonismo en la creación y el desborde, centrándose en culminar las jugadas.

2. **SUPPORT**: extremo de apoyo con menor protagonismo en la finalización y el riesgo. Contribuye al juego ofensivo mediante la circulación de balón y la asociación con compañeros, priorizando la conservación de la posesión sobre la acción individual.

In [29]:
# Diccionario con los nombres de los jugadores
player_names_dict = dict(zip(player_info_df["ID"], player_info_df["Name"]))

# Posición
pos = "WG"

# Seleccionamos posicion y aplicamos nombre del jugador
pos_df = clustered_dict[pos]
pos_df["PlayerName"] = pos_df.index.map(player_names_dict)
loadings_df = loadings_dict[pos]

# Carpeta destino
output_dir = os.path.join(os.getcwd(), "results", "visualizations", pos.lower())

# Si no existe, creamos
if not os.path.exists(output_dir):

    os.makedirs(output_dir)

    # Grupos
    components = loadings_df.columns
    c1 = components[0]
    c2 = components[1]
    c3 = components[2]
    c4 = components[3]
    c5 = components[4]

    # Creación de las visualizaciones
    fig_c1_c2 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c2, title=f"{c1} vs. {c2}")  # PC1-PC2
    fig_c1_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c3, title=f"{c1} vs. {c3}")  # PC1-PC3
    fig_c1_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c4, title=f"{c1} vs. {c4}")  # PC1-PC4
    fig_c1_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c5, title=f"{c1} vs. {c5}")  # PC1-PC5
    fig_c2_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c3, title=f"{c2} vs. {c3}")  # PC2-PC3
    fig_c2_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c4, title=f"{c2} vs. {c4}")  # PC2-PC4
    fig_c2_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c5, title=f"{c2} vs. {c5}")  # PC2-PC5
    fig_c3_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c4, title=f"{c3} vs. {c4}")  # PC3-PC4
    fig_c3_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c5, title=f"{c3} vs. {c5}")  # PC3-PC5
    fig_c4_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c4, y_col=c5, title=f"{c4} vs. {c5}")  # PC4-PC5

    # Nombres para cada gráfico
    names = [f"01_{c1}_{c2}", f"02_{c1}_{c3}", f"03_{c1}_{c4}", f"04_{c1}_{c5}", f"05_{c2}_{c3}", f"06_{c2}_{c4}", 
            f"07_{c2}_{c5}", f"08_{c3}_{c4}", f"09_{c3}_{c5}", f"10_{c4}_{c5}"]

    # Lista
    figs = [fig_c1_c2, fig_c1_c3, fig_c1_c4, fig_c1_c5, fig_c2_c3, fig_c2_c4, fig_c2_c5, fig_c3_c4, fig_c3_c5, fig_c4_c5]
    for fig, name in zip(figs, names):
        file_path = os.path.join(output_dir, f"{name}.png")
        fig.write_image(file_path, width=1200, height=800)

    # Creación del dataframe de pases
    fig, ax = plot_pitch_clusters(df_pos=pos_df, title=f"Average pass position of {pos} (clustered)")
    fig.savefig(os.path.join(output_dir, "PassField.png"), dpi=300, bbox_inches="tight")

#### **4.1.7. Delanteros**

0. **LINKER**: delantero con alta participación en el juego ofensivo. Destaca por su capacidad para intervenir en la creación, generar ocasiones mediante pases clave y conectar con los compañeros. Su rol no se limita a la finalización, sino que actúa como nexo entre líneas.

1. **POACHER**: delantero especializado en la finalización dentro del área. Presenta menor volumen de participación, pero se posiciona en zonas de remate, priorizando la eficiencia en el gol y la capacidad de aprovechar oportunidades.

2. **TARGET**: delantero de referencia física, orientado al juego directo. Sobresale en duelos, juego aéreo y acciones de contacto, siendo clave en situaciones de balón largo y en la fijación de defensores.

In [30]:
# Diccionario con los nombres de los jugadores
player_names_dict = dict(zip(player_info_df["ID"], player_info_df["Name"]))

# Posición
pos = "ST"

# Seleccionamos posicion y aplicamos nombre del jugador
pos_df = clustered_dict[pos]
pos_df["PlayerName"] = pos_df.index.map(player_names_dict)
loadings_df = loadings_dict[pos]

# Carpeta destino
output_dir = os.path.join(os.getcwd(), "results", "visualizations", pos.lower())

# Si no existe, creamos
if not os.path.exists(output_dir):

    os.makedirs(output_dir)

    # Grupos
    components = loadings_df.columns
    c1 = components[0]
    c2 = components[1]
    c3 = components[2]
    c4 = components[3]
    c5 = components[4]

    # Creación de las visualizaciones
    fig_c1_c2 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c2, title=f"{c1} vs. {c2}")  # PC1-PC2
    fig_c1_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c3, title=f"{c1} vs. {c3}")  # PC1-PC3
    fig_c1_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c4, title=f"{c1} vs. {c4}")  # PC1-PC4
    fig_c1_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c1, y_col=c5, title=f"{c1} vs. {c5}")  # PC1-PC5
    fig_c2_c3 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c3, title=f"{c2} vs. {c3}")  # PC2-PC3
    fig_c2_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c4, title=f"{c2} vs. {c4}")  # PC2-PC4
    fig_c2_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c2, y_col=c5, title=f"{c2} vs. {c5}")  # PC2-PC5
    fig_c3_c4 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c4, title=f"{c3} vs. {c4}")  # PC3-PC4
    fig_c3_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c3, y_col=c5, title=f"{c3} vs. {c5}")  # PC3-PC5
    fig_c4_c5 = pca_plot(df=pos_df, loadings_df=loadings_df, x_col=c4, y_col=c5, title=f"{c4} vs. {c5}")  # PC4-PC5

    # Nombres para cada gráfico
    names = [f"01_{c1}_{c2}", f"02_{c1}_{c3}", f"03_{c1}_{c4}", f"04_{c1}_{c5}", f"05_{c2}_{c3}", f"06_{c2}_{c4}", 
            f"07_{c2}_{c5}", f"08_{c3}_{c4}", f"09_{c3}_{c5}", f"10_{c4}_{c5}"]

    # Lista
    figs = [fig_c1_c2, fig_c1_c3, fig_c1_c4, fig_c1_c5, fig_c2_c3, fig_c2_c4, fig_c2_c5, fig_c3_c4, fig_c3_c5, fig_c4_c5]
    for fig, name in zip(figs, names):
        file_path = os.path.join(output_dir, f"{name}.png")
        fig.write_image(file_path, width=1200, height=800)

    # Creación del dataframe de pases
    fig, ax = plot_pitch_clusters(df_pos=pos_df, title=f"Average pass position of {pos} (clustered)")
    fig.savefig(os.path.join(output_dir, "PassField.png"), dpi=300, bbox_inches="tight")

### **4.2. Aplicado de los roles por posición**

Una vez se han decidido los siguientes roles según las posiciones:

* **Portero**: *Distributor* / *Line Keeper* / *Direct Outlet*
* **Central**: *Circulator* / *Deep Anchor* / *Fragile Defender* / *Duel Specialist* / *Complete Builder* / *Progressive Playmaker*
* **Lateral**: *Balanced* / *Runner* / *Defensive*
* **Centrocampista defensivo**: *Ball Winner* / *Deep Playmaker* / *Anchor*
* **Centrocampista ofensivo**: *Creator* / *Scorer* / *Hybrid*
* **Extremo**: *Explosive* / *Finisher* / *Support*
* **Delantero**: *Linker* / *Poacher* / *Target*

Se van a aplicar estos grupos al conjunto de datos *clusterizado*.

In [31]:
# Diccionario con los nuevos valores
dict_position_roles = {"GK": {0: "Distributor", 1: "Line Keeper", 2: "Direct Outlet"},
                       "CB": {0: "Circulator", 1: "Deep Anchor", 2: "Fragile Defender", 3: "Duel Specialist", 4: "Complete Builder", 5: "Progressive Playmaker"},
                       "FB": {0: "Balanced", 1: "Runner", 2: "Defensive"},
                       "DM": {0: "Ball Winner", 1: "Deep Playmaker", 2: "Anchor"},
                       "AM": {0: "Creator", 1: "Scorer", 2: "Hybrid"},
                       "WG": {0: "Explosive", 1: "Finisher", 2: "Support"},
                       "ST": {0: "Linker", 1: "Poacher", 2: "Target"}}

# Lista para añadir la información
list_final_roles = []

# Aplicamos pca para cada posición
for position in ["GK","CB","FB","DM","AM","WG","ST"]:

    # Seleccionamos dataframe con posición
    pos_df = clustered_dict[position]
    pos_df = pos_df[["Cluster"]]                            # Únicamente nos interesa el cluster para mapear
    pos_df.insert(1, "Position", position)                  # Añadimos posición

    # Diccionario con los roles por posición y aplicamos
    dict_roles = dict_position_roles[position]
    pos_df["Role"] = pos_df["Cluster"].map(dict_roles)

    # Sacamos columna de cluster y insertamos a la lista
    pos_df = pos_df.drop(columns="Cluster")
    list_final_roles.append(pos_df)

# Dataframe final con los roles
final_roles_df = pd.concat(list_final_roles)
final_roles_df = final_roles_df.reset_index().sort_values(by="Player").reset_index(drop=True)

---

## **5. Función de aplicado a los jugadores**

Finalmente se construye una función que, dado un jugador, lo proyecta sobre los componentes de su posición y le asigna el **cluster/rol** correspondiente.

> **Salida del estudio:** un **rol de juego** por jugador (p. ej. *"CB — COMPLETE BUILDER"*), derivado objetivamente de sus estadísticas mediante PCA + K-means. 

In [32]:
# Diccionario para guardar la información
all_positions_roles_averages = {}

# Aplicamos pca para cada posición
for position in ["GK","CB","FB","DM","AM","WG","ST"]:

    # Dataframes necesarios
    pos_stats_df = dict_position_stats[position].copy()                                     # Dataframe con estadísticas por jugador
    players_roles = final_roles_df[final_roles_df["Position"] == position]                  # Roles por futbolista de la posición

    # Diccionario y aplicado
    dict_players_roles = dict(zip(players_roles["Player"], players_roles["Role"]))
    new_pos_stats = pos_stats_df.copy()
    new_pos_stats.insert(0, "Role", new_pos_stats.index.map(dict_players_roles))

    # Sacamos los valores nan
    new_pos_stats = new_pos_stats.dropna(subset="Role")
    new_pos_stats = new_pos_stats.reset_index(drop=True)

    # Media por rol
    role_means = (new_pos_stats.groupby("Role").mean(numeric_only=True).reset_index())      # A dataframe
    role_dict = (role_means.set_index("Role").round(6).to_dict(orient="index"))             # A diccionario, para guardar

    # Aplicamos al diccionario
    all_positions_roles_averages[position] = role_dict

# Guardar en JSON el diccionario
with open(os.path.join(os.getcwd(), "results", "all_positions_roles_averages.json"), "w", encoding="utf-8") as f:
    json.dump(all_positions_roles_averages, f, ensure_ascii=False, indent=4)

In [33]:
# Función para obtener el mejor rol de un jugador
def best_role_for_player(player_last_games_stats: pd.DataFrame) -> str:

    player = player_last_games_stats.copy()

    # Información necesaria
    position = player["Position"].iloc[0]
    roles_dict = all_positions_roles_averages[position]

    # Columnas correctas = nombres métricas del primer rol
    cols_to_study = list(next(iter(roles_dict.values())).keys())

    # Selección columnas jugador
    player_row = player[cols_to_study].copy()

    # Vector jugador
    player_num_list = np.array(player_row.iloc[0].tolist(), dtype=float)

    # Distancias por rol
    roles_distance = {}

    for role, metrics in roles_dict.items():
        values_list = np.array(list(metrics.values()), dtype=float)
        dist = np.linalg.norm(player_num_list - values_list)
        roles_distance[role] = dist

    # Mejor rol
    best_role = min(roles_distance, key=roles_distance.get)

    return best_role

In [34]:
# Función para obtener el rol de un jugador
def obtain_player_role(last_matches_stats: pd.DataFrame, print_info: bool = True) -> pd.DataFrame:

    # Aplicamos posiciones generales e insertamos rol
    last_matches_stats["Position"] = np.where(last_matches_stats["Position"].isin(["LB", "RB", "LW", "RW", "CM"]), last_matches_stats["Position"].map({"LB":"FB", "RB":"FB", "LW":"WG", "RW":"WG", "CM":"DM"}), last_matches_stats["Position"])
    last_matches_stats.insert(2, "Role", None)

    total_players = len(last_matches_stats)
    i = 1

    # Para cada jugador
    for index, row in last_matches_stats.iterrows():

        # Obtenemos información necesaria y aplicamos función
        player = last_matches_stats.loc[[index]]
        player_role = best_role_for_player(player_last_games_stats=player)
        last_matches_stats.at[index, "Role"] = player_role        # Guardar resultado

        if print_info:
            print(f"Obtaining the player role of player {player['Player'].iloc[0]} [{i}/{total_players}] ({round(100*(i/total_players), 2)}%)      ", flush=True, end="\r")
            i += 1

    # Return únicamente del jugador, posición y rol
    last_matches_stats = last_matches_stats.reset_index()
    return last_matches_stats[["Player", "Position", "Role"]].copy()

In [37]:
# Lectura del dataframe
last_matches_stats = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "agg_player.csv"), sep=";").head(100)

# Obtenemos roles de jugadores
players_roles_df = obtain_player_role(last_matches_stats=last_matches_stats)
display(players_roles_df)


,Player,Position,Role
0,00D0J,CB,Circulator
1,00VQ6,ST,Linker
2,0131N,WG,Explosive
3,014CK,AM,Hybrid
4,01N61,ST,Linker
...,...,...,...
95,0BQBE,AM,Scorer
96,0BRLG,CB,Circulator
97,0BYY7,CB,Circulator
98,0C4UW,CB,Circulator


---